# Self-play throughput benchmark (Colab)

Measures **games/hour** across a grid of `(workers, games)` regimes. Calls self-play directly — **no training, no shards, no checkpoints** — only an append-only CSV on Drive.

**Grid:** `workers=1` and `workers=2` on games {32, 64, 128, 192} (256 skipped). Constant: 200 MCTS sims, net from `latest.pt`, Syzygy on.

**Run cells 1→8 top to bottom.** Set runtime to **GPU** first. Requires `latest.pt` on Drive.

Results → `CKPT_DIR/benchmark_throughput.csv`. Re-run cell 7 after disconnect: **always redoes warmup** (compile cache is per-session), then runs any pending timed rows.

In [ ]:
# 1. Get the code. Safe to re-run (always works from /content, never nests).
import os
%cd /content
if not os.path.exists('/content/immortalite-zero'):
    !git clone https://github.com/carlo-wong/immortalite-zero.git
%cd /content/immortalite-zero
!git pull --quiet
print('working dir:', os.getcwd())

In [ ]:
# 2. Install dependencies (Colab already has a CUDA torch build).
!pip install -q python-chess numpy tqdm pandas matplotlib

In [ ]:
# 3. Mount Google Drive for checkpoints and benchmark CSV.
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/immortalite_zero_checkpoints'
import os
os.makedirs(CKPT_DIR, exist_ok=True)
LATEST_PT = os.path.join(CKPT_DIR, 'latest.pt')
print('checkpoints ->', CKPT_DIR)
if os.path.exists(LATEST_PT):
    print('latest.pt found — benchmark will load production weights')
else:
    raise FileNotFoundError(f'Missing {LATEST_PT} — upload or train a checkpoint first')

In [ ]:
# 4. Confirm GPU and record device name for the CSV.
import torch

HAS_CUDA = torch.cuda.is_available()
if not HAS_CUDA:
    raise RuntimeError('GPU required for this benchmark — set Runtime → GPU')
DEVICE = 'cuda'
DEVICE_NAME = torch.cuda.get_device_name(0)
IS_T4 = 'T4' in DEVICE_NAME

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

print('CUDA available:', HAS_CUDA)
print('device:', DEVICE_NAME)
print('is_t4:', IS_T4)

In [ ]:
# 5. Syzygy 3-4-5 WDL — use Drive copy (CKPT_DIR/syzygy345) if present, else download.
import os
import re
import shutil
import urllib.parse
import urllib.request

if 'CKPT_DIR' not in globals():
    raise RuntimeError('Run cell 3 first (mount Drive).')

TB_DRIVE = os.path.join(CKPT_DIR, 'syzygy345')
TB_DIR = '/content/syzygy345'
TB_MIRRORS = [
    'http://tablebase.sesse.net/syzygy/3-4-5/',
]
os.makedirs(TB_DIR, exist_ok=True)
os.makedirs(TB_DRIVE, exist_ok=True)

index_html = None
TB_BASE_URL = None
for mirror in TB_MIRRORS:
    try:
        index_html = urllib.request.urlopen(mirror, timeout=30).read().decode('utf-8', errors='ignore')
        TB_BASE_URL = mirror
        break
    except Exception as exc:
        print(f'Failed mirror {mirror}: {exc}')

if index_html is None or TB_BASE_URL is None:
    raise RuntimeError('Could not reach any Syzygy mirror for 3-4-5 WDL files.')

files = sorted(set(re.findall(r'href="([^"/]+\.rtbw)"', index_html)))
if not files:
    raise RuntimeError(f'No .rtbw files found at Syzygy source URL: {TB_BASE_URL}')

drive_count = len([f for f in os.listdir(TB_DRIVE) if f.endswith('.rtbw')])
print(f'Syzygy on Drive -> {TB_DRIVE} ({drive_count} .rtbw files)')
print(f'Expected files: {len(files)}')

copied_from_drive = 0
downloaded = 0
for idx, name in enumerate(files, start=1):
    dst_local = os.path.join(TB_DIR, name)
    dst_drive = os.path.join(TB_DRIVE, name)
    if os.path.exists(dst_local):
        continue
    if os.path.exists(dst_drive):
        shutil.copy2(dst_drive, dst_local)
        copied_from_drive += 1
        continue
    src = urllib.parse.urljoin(TB_BASE_URL, name)
    urllib.request.urlretrieve(src, dst_local)
    shutil.copy2(dst_local, dst_drive)
    downloaded += 1
    if idx % 25 == 0 or idx == len(files):
        print(f'  progress {idx}/{len(files)}')

local_count = len([f for f in os.listdir(TB_DIR) if f.endswith('.rtbw')])
print(f'Copied from Drive: {copied_from_drive}, downloaded: {downloaded}')
print(f'Syzygy ready -> {TB_DIR} ({local_count}/{len(files)} files)')
if local_count < len(files):
    raise RuntimeError('Syzygy setup incomplete — check Drive upload or network.')

In [ ]:
# 6. Benchmark grid and output path.
import os

if 'CKPT_DIR' not in globals():
    raise RuntimeError('Run cell 3 first.')

PLATFORM = 'colab'
SIMS = 200
GAMES_GRID = [32, 64, 128, 192, 256]
WORKERS1_GAMES = [32, 64, 128, 192]  # skip 256 — higher concurrency didn't raise games/hr
WORKERS2_GAMES = [32, 64, 128, 192]  # skip 256 for workers=2 as well
WORKERS_GRID = [1, 2]
TIMED_RUNS = 2
WARMUP_MOVES = 8
CSV_PATH = os.path.join(CKPT_DIR, 'benchmark_throughput.csv')

print('platform:', PLATFORM)
print('sims:', SIMS)
print('workers=1 games:', WORKERS1_GAMES)
print('workers=2 games:', WORKERS2_GAMES)
print('timed runs per config:', TIMED_RUNS)
print('warmup max moves:', WARMUP_MOVES)
print('csv ->', CSV_PATH)

In [ ]:
# 7. Run benchmark grid (self-play only — no shards, no checkpoints).
import csv
import os
import statistics
import time

import chess.syzygy
import torch

from engine.config import Config, NetConfig
from engine.encoding import ENCODING_VERSION
from engine.network import ChessNet, NetEvaluator
from engine.selfplay import GameResult, play_games_batched, play_games_parallel
from engine.train import _load_matching_state_dict

for name in ('PLATFORM', 'SIMS', 'WORKERS1_GAMES', 'WORKERS2_GAMES', 'WORKERS_GRID',
             'TIMED_RUNS', 'WARMUP_MOVES', 'CSV_PATH', 'LATEST_PT', 'TB_DIR', 'DEVICE', 'DEVICE_NAME'):
    if name not in globals():
        raise RuntimeError(f'Run setup cells first — missing {name}')

CSV_HEADER = (
    'platform,device,net_blocks,net_filters,sims,workers,games,'
    'per_worker_concurrency,syzygy,run_index,is_warmup,elapsed_seconds,'
    'games_completed,sec_per_game,games_per_hour,mean_game_len,total_positions\n'
)
REAL_MAX_MOVES = 200
StepKey = tuple[int, int, int, int]  # workers, games, is_warmup, run_index


def append_benchmark_row(path: str, row: dict) -> None:
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    new = not os.path.exists(path)
    with open(path, 'a', encoding='utf-8') as f:
        if new:
            f.write(CSV_HEADER)
        f.write(
            f"{row['platform']},{row['device']},"
            f"{row['net_blocks']},{row['net_filters']},{row['sims']},"
            f"{row['workers']},{row['games']},{row['per_worker_concurrency']},"
            f"{row['syzygy']},{row['run_index']},{row['is_warmup']},"
            f"{row['elapsed_seconds']:.3f},{row['games_completed']},"
            f"{row['sec_per_game']:.4f},{row['games_per_hour']:.2f},"
            f"{row['mean_game_len']:.4f},{row['total_positions']}\n"
        )


def load_completed_steps(path: str) -> set[StepKey]:
    if not os.path.exists(path):
        return set()
    completed: set[StepKey] = set()
    with open(path, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for r in reader:
            completed.add((
                int(r['workers']),
                int(r['games']),
                int(r['is_warmup']),
                int(r['run_index']),
            ))
    return completed


def step_done(completed: set[StepKey], workers: int, games: int,
              is_warmup: int, run_index: int) -> bool:
    return (workers, games, is_warmup, run_index) in completed


def config_fully_done(completed: set[StepKey], workers: int, games: int) -> bool:
    return all(
        step_done(completed, workers, games, 0, run_idx)
        for run_idx in range(1, TIMED_RUNS + 1)
    )


def pending_timed_runs(completed: set[StepKey], workers: int, games: int) -> list[int]:
    return [
        run_idx for run_idx in range(1, TIMED_RUNS + 1)
        if not step_done(completed, workers, games, 0, run_idx)
    ]


def games_for_workers(workers: int) -> list[int]:
    return sorted(WORKERS1_GAMES) if workers == 1 else list(WORKERS2_GAMES)


def next_pending_label(completed: set[StepKey]) -> str | None:
    for workers in WORKERS_GRID:
        for games in games_for_workers(workers):
            pending = pending_timed_runs(completed, workers, games)
            if pending:
                return (
                    f'workers={workers} games={games} '
                    f'session warmup then timed {pending[0]}/{TIMED_RUNS}'
                )
    return None


def run_session_warmup(
    completed: set[StepKey],
    *,
    workers: int,
    games: int,
    per_wc: int,
    run_fn,
) -> None:
    """Always run warmup before pending timed runs (compile/cudnn are per-session)."""
    cfg.train.max_game_moves = WARMUP_MOVES
    elapsed, lengths = run_fn(games, SIMS)
    if step_done(completed, workers, games, 1, 0):
        print(
            f'  session warmup: {elapsed:.1f}s ({len(lengths)} games) '
            f'— not logged (CSV row from prior session)'
        )
    else:
        log_step(completed, make_row(
            platform=PLATFORM, device=DEVICE_NAME, net_cfg=cfg.net, sims=SIMS,
            workers=workers, games=games, per_worker_concurrency=per_wc, syzygy=1,
            run_index=0, is_warmup=1, elapsed=elapsed, game_lengths=lengths,
        ))
        print(f'  warmup: {elapsed:.1f}s ({len(lengths)} games, max_moves={WARMUP_MOVES})')


def make_row(*, platform, device, net_cfg, sims, workers, games, per_worker_concurrency,
             syzygy, run_index, is_warmup, elapsed, game_lengths) -> dict:
    completed = len(game_lengths)
    sec_per_game = elapsed / completed if completed else float('nan')
    games_per_hour = 3600.0 / sec_per_game if completed and sec_per_game > 0 else float('nan')
    mean_len = statistics.mean(game_lengths) if game_lengths else float('nan')
    total_positions = sum(game_lengths)
    return {
        'platform': platform,
        'device': device,
        'net_blocks': net_cfg.blocks,
        'net_filters': net_cfg.filters,
        'sims': sims,
        'workers': workers,
        'games': games,
        'per_worker_concurrency': per_worker_concurrency,
        'syzygy': syzygy,
        'run_index': run_index,
        'is_warmup': is_warmup,
        'elapsed_seconds': elapsed,
        'games_completed': completed,
        'sec_per_game': sec_per_game,
        'games_per_hour': games_per_hour,
        'mean_game_len': mean_len,
        'total_positions': total_positions,
    }


def run_workers1(evaluator, cfg, tablebase, games: int, sims: int) -> tuple[float, list[int]]:
    game_lengths: list[int] = []

    def _on_game(game: GameResult) -> None:
        game_lengths.append(len(game.samples))

    t0 = time.time()
    play_games_batched(
        evaluator,
        cfg,
        simulations=sims,
        num_games=games,
        concurrency=games,
        on_game_finished=_on_game,
        tablebase=tablebase,
    )
    return time.time() - t0, game_lengths


def run_workers2(cfg, weights_path: str, games: int, workers: int, sims: int,
                 syzygy_path: str) -> tuple[float, list[int]]:
    t0 = time.time()
    _, _, game_lengths, _ = play_games_parallel(
        cfg,
        cfg.net,
        weights_path,
        simulations=sims,
        num_games=games,
        workers=workers,
        device=DEVICE,
        syzygy_path=syzygy_path,
    )
    return time.time() - t0, game_lengths


def log_step(completed: set[StepKey], row: dict) -> None:
    append_benchmark_row(CSV_PATH, row)
    completed.add((
        int(row['workers']),
        int(row['games']),
        int(row['is_warmup']),
        int(row['run_index']),
    ))


state = torch.load(LATEST_PT, map_location=DEVICE)
ckpt_encoding = int(state.get('encoding_version', 1)) if isinstance(state, dict) else 1
if ckpt_encoding != ENCODING_VERSION:
    raise ValueError(
        f'checkpoint encoding {ckpt_encoding} != current {ENCODING_VERSION}'
    )

cfg = Config()
if isinstance(state, dict) and 'net' in state:
    cfg.net = NetConfig(**state['net'])
cfg.train.syzygy_path = TB_DIR
cfg.train.draw_penalty = 1 / 3
cfg.mcts.draw_contempt = cfg.train.draw_penalty
cfg.train.sims_per_move = SIMS
cfg.mcts.simulations = SIMS

tablebase = chess.syzygy.open_tablebase(TB_DIR)
net = ChessNet(cfg.net).to(DEVICE)
model_state = state['model'] if isinstance(state, dict) and 'model' in state else state
_load_matching_state_dict(net, model_state, label='benchmark load', verbose=True)
if hasattr(torch, 'compile'):
    net = torch.compile(net, dynamic=True)
else:
    print('warning: torch.compile unavailable — timing without compile')
evaluator = NetEvaluator(net, device=DEVICE)

completed_steps = load_completed_steps(CSV_PATH)
print(f'loaded net {cfg.net.blocks}x{cfg.net.filters}, sims={SIMS}')
print(f'resume: {len(completed_steps)} steps already in CSV')
pending = next_pending_label(completed_steps)
if pending:
    print(f'next pending step: {pending}')
else:
    print('all steps complete — nothing to run')

try:
    # workers=1: ascending games so compile/cudnn caches carry over
    for games in sorted(WORKERS1_GAMES):
        workers = 1
        pending = pending_timed_runs(completed_steps, workers, games)
        if not pending:
            print(f'skip workers=1 games={games} (timed runs complete)')
            continue
        per_wc = games
        print(f'\n=== workers=1 games={games} concurrency={games} ===')
        run_session_warmup(
            completed_steps,
            workers=workers,
            games=games,
            per_wc=per_wc,
            run_fn=lambda g, s: run_workers1(evaluator, cfg, tablebase, g, s),
        )

        cfg.train.max_game_moves = REAL_MAX_MOVES
        for run_idx in pending:
            elapsed, lengths = run_workers1(evaluator, cfg, tablebase, games, SIMS)
            row = make_row(
                platform=PLATFORM, device=DEVICE_NAME, net_cfg=cfg.net, sims=SIMS,
                workers=workers, games=games, per_worker_concurrency=per_wc, syzygy=1,
                run_index=run_idx, is_warmup=0, elapsed=elapsed, game_lengths=lengths,
            )
            log_step(completed_steps, row)
            print(
                f'  timed run {run_idx}/{TIMED_RUNS}: '
                f"{row['games_per_hour']:.0f} games/hr "
                f"({row['sec_per_game']:.2f} s/game, mean_len={row['mean_game_len']:.1f})"
            )

    # workers=2: separate CUDA contexts per worker (no torch.compile in workers)
    for games in WORKERS2_GAMES:
        workers = 2
        pending = pending_timed_runs(completed_steps, workers, games)
        if not pending:
            print(f'skip workers=2 games={games} (timed runs complete)')
            continue
        per_wc = games // workers
        print(f'\n=== workers=2 games={games} per_worker_concurrency={per_wc} ===')
        run_session_warmup(
            completed_steps,
            workers=workers,
            games=games,
            per_wc=per_wc,
            run_fn=lambda g, s: run_workers2(cfg, LATEST_PT, g, workers, s, TB_DIR),
        )

        cfg.train.max_game_moves = REAL_MAX_MOVES
        for run_idx in pending:
            elapsed, lengths = run_workers2(cfg, LATEST_PT, games, workers, SIMS, TB_DIR)
            row = make_row(
                platform=PLATFORM, device=DEVICE_NAME, net_cfg=cfg.net, sims=SIMS,
                workers=workers, games=games, per_worker_concurrency=per_wc, syzygy=1,
                run_index=run_idx, is_warmup=0, elapsed=elapsed, game_lengths=lengths,
            )
            log_step(completed_steps, row)
            print(
                f'  timed run {run_idx}/{TIMED_RUNS}: '
                f"{row['games_per_hour']:.0f} games/hr "
                f"({row['sec_per_game']:.2f} s/game, mean_len={row['mean_game_len']:.1f})"
            )
finally:
    tablebase.close()

print(f'\nDone. Results -> {CSV_PATH}')

In [ ]:
# 8. Aggregate results (exclude warmups) and plot games/hour.
import os

import matplotlib.pyplot as plt
import pandas as pd

if 'CSV_PATH' not in globals():
    raise RuntimeError('Run cell 6 first.')

if not os.path.exists(CSV_PATH):
    print(f'No results yet at {CSV_PATH}. Run cell 7 first.')
else:
    df = pd.read_csv(CSV_PATH)
    for col in df.columns:
        if col not in ('platform', 'device'):
            df[col] = pd.to_numeric(df[col], errors='coerce')

    timed = df[df['is_warmup'] == 0].copy()
    if timed.empty:
        print('CSV exists but no timed rows yet.')
    else:
        summary = (
            timed.groupby(['workers', 'games'], as_index=False)
            .agg(
                median_sec_per_game=('sec_per_game', 'median'),
                median_games_per_hour=('games_per_hour', 'median'),
                median_mean_game_len=('mean_game_len', 'median'),
                runs=('run_index', 'count'),
            )
            .sort_values(['workers', 'games'])
        )
        print('Median throughput (timed runs only):')
        print(summary.to_string(index=False))

        best = summary.loc[summary['median_games_per_hour'].idxmax()]
        print(
            f"\nBest: workers={int(best['workers'])} games={int(best['games'])} "
            f"-> {best['median_games_per_hour']:.0f} games/hr "
            f"({best['median_sec_per_game']:.2f} s/game)"
        )

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        for workers, sub in summary.groupby('workers'):
            axes[0].plot(
                sub['games'], sub['median_games_per_hour'],
                marker='o', label=f'workers={int(workers)}',
            )
        axes[0].set_xlabel('games (= concurrency)')
        axes[0].set_ylabel('median games/hour')
        axes[0].set_title('Self-play throughput')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        for workers, sub in summary.groupby('workers'):
            axes[1].plot(
                sub['games'], sub['median_sec_per_game'],
                marker='o', label=f'workers={int(workers)}',
            )
        axes[1].set_xlabel('games (= concurrency)')
        axes[1].set_ylabel('median sec/game')
        axes[1].set_title('Seconds per game')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()